In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
df = pd.read_csv('dataset/train.csv')
df.describe()
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517754 entries, 0 to 517753
Data columns (total 14 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      517754 non-null  int64  
 1   road_type               517754 non-null  object 
 2   num_lanes               517754 non-null  int64  
 3   curvature               517754 non-null  float64
 4   speed_limit             517754 non-null  int64  
 5   lighting                517754 non-null  object 
 6   weather                 517754 non-null  object 
 7   road_signs_present      517754 non-null  bool   
 8   public_road             517754 non-null  bool   
 9   time_of_day             517754 non-null  object 
 10  holiday                 517754 non-null  bool   
 11  school_season           517754 non-null  bool   
 12  num_reported_accidents  517754 non-null  int64  
 13  accident_risk           517754 non-null  float64
dtypes: bool(4), float64(

id                        0
road_type                 0
num_lanes                 0
curvature                 0
speed_limit               0
lighting                  0
weather                   0
road_signs_present        0
public_road               0
time_of_day               0
holiday                   0
school_season             0
num_reported_accidents    0
accident_risk             0
dtype: int64

In [13]:
onehotdf = pd.get_dummies(df, drop_first=True)
onehotdf = onehotdf.drop(columns=['id'])

In [14]:
correlation = onehotdf.corr()['accident_risk'].sort_values(ascending=False)
print(correlation)

accident_risk             1.000000
curvature                 0.543946
lighting_night            0.465798
speed_limit               0.430898
num_reported_accidents    0.213891
weather_foggy             0.149758
holiday                   0.051129
weather_rainy             0.036137
public_road               0.031032
road_type_urban           0.021463
time_of_day_evening       0.010032
road_signs_present        0.000629
school_season            -0.000977
num_lanes                -0.006003
time_of_day_morning      -0.006019
road_type_rural          -0.010121
lighting_dim             -0.233032
Name: accident_risk, dtype: float64


In [15]:
df_features = onehotdf.drop(columns=['accident_risk'])
numerical_cols = df_features.columns.tolist()
engineered_features_df = pd.DataFrame()

for col in numerical_cols:
    engineered_features_df[f'{col}_log'] = np.log1p(df_features[col])
    engineered_features_df[f'{col}_sq'] = df_features[col] ** 2

for i in range(len(numerical_cols)):
    for j in range(i + 1, len(numerical_cols)):
        col1 = numerical_cols[i]
        col2 = numerical_cols[j]
        feature_name = f'{col1}_x_{col2}'
        engineered_features_df[feature_name] = df_features[col1] * df_features[col2]
        print(f"Created interaction feature: {feature_name}")

print(engineered_features_df.head())

Created interaction feature: num_lanes_x_curvature
Created interaction feature: num_lanes_x_speed_limit
Created interaction feature: num_lanes_x_road_signs_present
Created interaction feature: num_lanes_x_public_road
Created interaction feature: num_lanes_x_holiday
Created interaction feature: num_lanes_x_school_season
Created interaction feature: num_lanes_x_num_reported_accidents
Created interaction feature: num_lanes_x_road_type_rural
Created interaction feature: num_lanes_x_road_type_urban
Created interaction feature: num_lanes_x_lighting_dim
Created interaction feature: num_lanes_x_lighting_night
Created interaction feature: num_lanes_x_weather_foggy
Created interaction feature: num_lanes_x_weather_rainy
Created interaction feature: num_lanes_x_time_of_day_evening
Created interaction feature: num_lanes_x_time_of_day_morning
Created interaction feature: curvature_x_speed_limit
Created interaction feature: curvature_x_road_signs_present
Created interaction feature: curvature_x_publi

C:\Users\tungl\AppData\Local\Temp\ipykernel_10780\3487593954.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  engineered_features_df[feature_name] = df_features[col1] * df_features[col2]
C:\Users\tungl\AppData\Local\Temp\ipykernel_10780\3487593954.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  engineered_features_df[feature_name] = df_features[col1] * df_features[col2]
C:\Users\tungl\AppData\Local\Temp\ipykernel_10780\3487593954.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of c

In [16]:
fulldf = pd.concat([onehotdf, engineered_features_df], axis=1)

In [17]:
best_df = fulldf

In [18]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, KFold
from scipy.stats import uniform, randint


In [19]:
X_train, X_test = train_test_split(best_df, test_size=0.2, random_state=42, stratify=df['accident_risk'])
y_train = X_train.pop('accident_risk')
y_test = X_test.pop('accident_risk')

In [20]:
model = xgb.XGBRegressor(tree_method='hist', random_state=42, earlier_stopping_rounds=50, device='cuda')

params = {
    'n_estimators': randint(100, 1500),
    'max_depth': randint(3, 8),
    'learning_rate': uniform(0.01, 0.1),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'gamma': [0, 0.5, 1],
    'reg_alpha': [0, 0.001, 0.005, 0.01]
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=50,
    scoring='neg_mean_squared_error',
    cv=kf,
    verbose=1,
    n_jobs=1,
    random_state=42
)
fit_params = {
    'eval_set' : [(X_test, y_test)],
    'verbose' : False
}

random_search.fit(X_train, y_train, **fit_params)
best_model = random_search.best_estimator_
best_params = random_search.best_params_
print("Best Hyperparameters:", best_params)
y_pred = best_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = mse**0.5
print(f'RMSE: {rmse}')

Fitting 5 folds for each of 50 candidates, totalling 250 fits


c:\Users\tungl\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:23:48] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "earlier_stopping_rounds" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\tungl\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\core.py:729: UserWarning: [13:24:15] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
c:\Users\tungl\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: Us

Best Hyperparameters: {'colsample_bytree': np.float64(0.8963074471016818), 'gamma': 0, 'learning_rate': np.float64(0.02366213314420288), 'max_depth': 7, 'n_estimators': 858, 'reg_alpha': 0.001, 'subsample': np.float64(0.9240453578716723)}
RMSE: 0.05628530969674099


In [21]:
test_df = pd.read_csv('dataset/test.csv')
test_ids = test_df['id']
test_df = pd.get_dummies(test_df, drop_first=True)
test_df = test_df.drop(columns=['id'])

numerical_cols = test_df.columns.tolist()
test_df = pd.DataFrame()

for col in numerical_cols:
    test_df[f'{col}_log'] = np.log1p(test_df[col])
    test_df[f'{col}_sq'] = test_df[col] ** 2

for i in range(len(numerical_cols)):
    for j in range(i + 1, len(numerical_cols)):
        col1 = numerical_cols[i]
        col2 = numerical_cols[j]
        feature_name = f'{col1}_x_{col2}'
        test_df[feature_name] = test_df[col1] * test_df[col2]

test_features = test_df[best_df.index.drop('accident_risk')]

submission_df = pd.DataFrame({
    'id': test_ids,
    'accident_risk': best_model.predict(test_features)
})

submission_df.to_csv('submission.csv', index=False)